# OceanHeart：亲眼看 Agent 怎样调用工具

选择 **OceanHeart Agents SDK** 内核，按顺序「全部运行」。只做本地推理，不训练、不调用在线 judge、不需要 API Key。

这里使用正式工具 SFT 的最佳 Dense 权重，而不是两步烟测权重。SDK 负责运行循环，海洋工具负责实际检索与计算，模型决定是否调用以及填写什么参数。

演示包含：计算、海洋资料检索、多工具串联、错误恢复、缺信息澄清、无需工具直接回答。**这是功能观察，不是新的完整基准；错误会原样展示。**

首次配置（终端执行；当前机器已配置）：
```bash
/home/anhuang/.venvs/ocean-agents-sdk/bin/pip install ipykernel nbformat nbclient
/home/anhuang/.venvs/ocean-agents-sdk/bin/python -m ipykernel install --user --name ocean-agents-sdk --display-name 'OceanHeart Agents SDK'
```


In [1]:
import asyncio
import json
import sys
import time
from pathlib import Path
from datetime import datetime
from importlib.metadata import version

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "agent/sdk.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请从 OceanHeart 项目或其子目录打开 Notebook")
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoTokenizer
from agent.ocean import TOOLS, read_jsonl, file_hash
from agent.react import ReactTools
from agent.sdk import run, run_trajectory
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from trainer.train_agent import model_generator, atomic_json

WEIGHT = ROOT / "out/ocean_react-v2-20260927-1046-sft_best_768.pth"
DATA = ROOT / "data/processed/react-v2"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"  # 可改为其他空闲显卡
for path in [WEIGHT, DATA / "ocean_agent_val.jsonl", DATA / "ocean_agent_corpus_val.jsonl"]:
    if not path.is_file():
        raise FileNotFoundError(f"缺少本地资产：{path}；Git 不包含权重或生成数据")

torch.set_num_threads(2)
torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained(ROOT / "model", local_files_only=True)
corpus = read_jsonl(DATA / "ocean_agent_corpus_val.jsonl")
tasks = read_jsonl(DATA / "ocean_agent_val.jsonl")
print("内核：", sys.executable)
print("SDK：", version("openai-agents"), "  设备：", DEVICE)
print("权重：", WEIGHT.name)


/home/anhuang/.conda/envs/minimind/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


/home/anhuang/.conda/envs/minimind/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


内核： /home/anhuang/.venvs/ocean-agents-sdk/bin/python
SDK： 0.22.3   设备： cuda:0
权重： ocean_react-v2-20260927-1046-sft_best_768.pth


## 1. 到底有哪些工具？

- `search_ocean(query)`：本地海洋资料／模拟航次卡检索，返回文段、编号和来源。本例只使用验证侧资料库。
- `marine_calculate(...)`：海里／公里、节／km/h 等单位换算；航程、航速和时间计算。

下面是**直接测试工具**，不是让模型回答。工具本身算对，不代表模型会正确调用它。模型没有 Shell、文件写入或任意代码执行工具。OceanInstruct 及模拟卡可能包含合成内容，不能当作真实航行决策依据。


In [2]:
for tool in TOOLS:
    print(tool["function"]["name"], "：", tool["function"]["description"])
env = ReactTools(corpus)
try:
    for arguments in [
        {"operation": "convert", "value": 1, "from_unit": "nmi", "to_unit": "km"},
        {"operation": "distance", "speed": 12, "hours": 3},
    ]:
        print(arguments, "→", env.execute({"name": "marine_calculate", "arguments": arguments}))
finally:
    env.db.close()


search_ocean ： 检索海洋资料或模拟航次卡，返回原文和编号。
marine_calculate ： convert换算；distance=速度×小时，time=距离/速度，speed=距离/小时。距离km，速度km/h，时间h。
{'operation': 'convert', 'value': 1, 'from_unit': 'nmi', 'to_unit': 'km'} → {'value': 1.852, 'unit': 'km'}
{'operation': 'distance', 'speed': 12, 'hours': 3} → {'value': 36.0, 'unit': 'km'}


## 2. 加载训练后的模型

固定贪心生成，不重试到成功、不修正模型的工具参数。默认最多 6 轮生成、4 次调用、2048 tokens 上下文。


In [3]:
device = torch.device(DEVICE)
model = MiniMindForCausalLM(MiniMindConfig(hidden_size=768, num_hidden_layers=8, use_moe=False))
model.load_state_dict(torch.load(WEIGHT, map_location="cpu", weights_only=True), strict=True)
model.to(device).eval()
generate = model_generator(model, tokenizer, device, sample=False, save_logps=False)
OUTPUT = ROOT / "artifacts/agent" / ("notebook-sdk-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
OUTPUT.mkdir(parents=True, exist_ok=False)
atomic_json(OUTPUT / "config.json", {
    "weight": str(WEIGHT), "weight_sha256": file_hash(WEIGHT),
    "corpus_sha256": file_hash(DATA / "ocean_agent_corpus_val.jsonl"),
    "adapter_sha256": file_hash(ROOT / "agent/sdk.py"),
    "sdk_version": version("openai-agents"), "device": DEVICE,
    "sample": False, "selection": "first validation task of each kind, no success filtering",
})
print("模型已加载。完整轨迹保存到：", OUTPUT)


模型已加载。完整轨迹保存到： /home/anhuang/OceanHeart/artifacts/agent/notebook-sdk-20260927-152635-804324


In [4]:
def run_case(item):
    # Notebook 已有事件循环；在一个工作线程运行现有同步 SDK 入口。
    # SQLite 连接在同一线程创建、使用和关闭；示例按顺序执行，不并发共享模型。
    env = ReactTools(corpus)
    try:
        if isinstance(item, str):
            return run(item, tokenizer, env, generate)
        return run_trajectory(item, tokenizer, env, generate)
    finally:
        env.db.close()

def show_trace(trace):
    call_index = 0
    for i, step in enumerate(trace["rounds"], 1):
        print(f"\n第 {i} 轮模型输出：")
        print(step["text"])
        if step.get("action", {}).get("type") in {"tool", "invalid"}:
            if call_index < len(trace["calls"]):
                call = trace["calls"][call_index]
                print("实际调用：", json.dumps(call["call"], ensure_ascii=False))
                print("工具结果：", json.dumps(call["result"], ensure_ascii=False))
                call_index += 1
            else:
                print("没有执行工具（请查看停止原因）")
    print("\n最终回答：", trace["final"] or "（未完成）")
    print("停止原因：", trace["stop"], "；调用尝试：", len(trace["calls"]))
    if "metrics" in trace:
        print("任务规则判定：", "成功" if trace["metrics"]["success"] else "失败")
        print("注意：final 仅表示生成结束，不代表答案正确。")


## 3. 观察六种行为

每类只取验证文件中出现的第一题，不按结果挑题。问题和公开历史交给 Agent，标准答案只在执行后交给原评分器。

「错误恢复」题可能注入一次明确标记的模拟工具故障；它不是线上工具随机失灵。「澄清」和「直接回答」可能无需调用工具。功能存在不等于每题都能完成。


In [5]:
labels = {"calculate": "计算", "retrieve": "检索与引用", "chain": "检索后计算",
          "recovery": "错误恢复", "clarify": "信息不足时澄清", "direct": "无需工具直接回答"}
traces = []
for kind, label in labels.items():
    task = next(t for t in tasks if t["kind"] == kind)
    print("\n" + "=" * 60)
    print(label, "｜", task["id"])
    print("问题：", task["question"])
    if task.get("public"):
        print("公开场景（模拟故障／历史）：", json.dumps(task["public"], ensure_ascii=False))
    started = time.perf_counter()
    trace = await asyncio.to_thread(run_case, task)
    trace["demo_seconds"] = time.perf_counter() - started
    traces.append(trace)
    atomic_json(OUTPUT / (task["id"] + ".json"), trace)
    show_trace(trace)



计算 ｜ val-f6b9dfb2c6189aef
问题： 海洋调查记录处理：已知航速6.2 km/h与航行时长1.7 h，求调查船走过的距离。只回答数值和单位。



第 1 轮模型输出：
<plan>先查找相关海洋资料。</plan><tool_call>{"name": "search_ocean", "arguments": {"query": "已知航速6.2 km/h与航行时长1.7 h"}}</tool_call>
实际调用： {"name": "search_ocean", "arguments": {"query": "已知航速6.2 km/h与航行时长1.7 h"}}
工具结果： {"documents": []}

第 2 轮模型输出：
<clarify>未找到航速6.2 km/h与航行时长1.7 h</clarify>

最终回答： 未找到航速6.2 km/h与航行时长1.7 h
停止原因： clarify ；调用尝试： 1
任务规则判定： 失败
注意：final 仅表示生成结束，不代表答案正确。

检索与引用 ｜ val-4ca1f30f8b2f6a96
问题： 海洋调查记录处理：请为问题‘宽壳全海笋的受精卵在发育过程中纺锤体的作用是什么？’找出资料中的一句原文，并附上[文档编号]。



第 1 轮模型输出：
<plan>先查找相关海洋资料。</plan><tool_call>{"name": "search_ocean", "arguments": {"query": "请为问题‘宽壳全海笋的受精卵在发育过程中纺锤体的作用是什么？"}}</tool_call>
实际调用： {"name": "search_ocean", "arguments": {"query": "请为问题‘宽壳全海笋的受精卵在发育过程中纺锤体的作用是什么？"}}
工具结果： {"documents": [{"id": "O6fb9da0f92", "source": "OceanInstruct-v0.2", "excerpt": "a. 染色体排列与分离：纺锤体在细胞分裂过程中形成，其纤维结构将染色体精确排列在赤道板（细胞中央平面）上，确保后续的均等分离。"}]}

第 2 轮模型输出：
<final>a. 染色体排列与分离：纺锤体在细胞分裂过程中形成，其纤维结构将染色体精确排列在赤道板（细胞中央平面）上，确保后续的均等分离。[O6fb9da0f92]</final>

最终回答： a. 染色体排列与分离：纺锤体在细胞分裂过程中形成，其纤维结构将染色体精确排列在赤道板（细胞中央平面）上，确保后续的均等分离。[O6fb9da0f92]
停止原因： final ；调用尝试： 1
任务规则判定： 成功
注意：final 仅表示生成结束，不代表答案正确。

检索后计算 ｜ val-759f449abd89be90
问题： 海洋调查记录处理：请根据模拟航次卡 C6e9eec9f15的航速和时长确定总里程，统一以km表示。只回答数值和单位。



第 1 轮模型输出：
<plan>先查找相关海洋资料。</plan><tool_call>{"name": "search_ocean", "arguments": {"query": "请根据模拟航次卡 C6e9eec9f15"}}</tool_call>
实际调用： {"name": "search_ocean", "arguments": {"query": "请根据模拟航次卡 C6e9eec9f15"}}
工具结果： {"documents": [{"id": "C6e9eec9f15", "source": "synthetic-v2", "excerpt": "合成练习，非实测：speed=39 kn，hours=5.4 h。"}]}

第 2 轮模型输出：
<plan>按已知数值与单位计算。</plan><tool_call>{"name": "marine_calculate", "arguments": {"operation": "convert", "value": 39.0, "from_unit": "kn", "to_unit": "km/h"}}</tool_call>
实际调用： {"name": "marine_calculate", "arguments": {"operation": "convert", "value": 39.0, "from_unit": "kn", "to_unit": "km/h"}}
工具结果： {"value": 72.228, "unit": "km/h"}

第 3 轮模型输出：
<plan>按已知数值与单位计算。</plan><tool_call>{"name": "marine_calculate", "arguments": {"operation": "distance", "speed": 72.228, "hours": 5.4}}</tool_call>
实际调用： {"name": "marine_calculate", "arguments": {"operation": "distance", "speed": 72.228, "hours": 5.4}}
工具结果： {"value": 390.0312, "unit": "km"}

第 4 轮模型输出：
<final>


第 1 轮模型输出：
<plan>先查找相关海洋资料。</plan><tool_call>{"name": "search_ocean", "arguments": {"query": "已知航速32.6 km/h与航行时长7.2 h"}}</tool_call>
实际调用： {"name": "search_ocean", "arguments": {"query": "已知航速32.6 km/h与航行时长7.2 h"}}
工具结果： {"documents": []}

第 2 轮模型输出：
<clarify>未找到航速32.6 km/h与航行时长7.2 h</clarify>

最终回答： 未找到航速32.6 km/h与航行时长7.2 h
停止原因： clarify ；调用尝试： 1
任务规则判定： 失败
注意：final 仅表示生成结束，不代表答案正确。

信息不足时澄清 ｜ val-ef857c4603305ade
问题： 海洋调查记录处理：请找出航次missingf31fb857fa的资料；无法定位时请告诉我需补充的信息。



第 1 轮模型输出：
<clarify>请找出航次missingf31fb857fa的资料；无法定位时请告诉我需补充的信息。</clarify>

最终回答： 请找出航次missingf31fb857fa的资料；无法定位时请告诉我需补充的信息。
停止原因： clarify ；调用尝试： 0
任务规则判定： 失败
注意：final 仅表示生成结束，不代表答案正确。

无需工具直接回答 ｜ val-63e1728ffdb6b392
问题： 海洋调查记录处理：请照录这份已确认海洋记录的内容：i. 由于其多孔结构提供了更大的表面积和生态位，通常支持更高多样性的微生物群落。 不要添加别的文字。



第 1 轮模型输出：
<final>i. 由于其多孔结构提供了更大的表面积和生态位，通常支持更高多样性的微生物群落。</final>

最终回答： i. 由于其多孔结构提供了更大的表面积和生态位，通常支持更高多样性的微生物群落。
停止原因： final ；调用尝试： 0
任务规则判定： 成功
注意：final 仅表示生成结束，不代表答案正确。


## 4. 汇总怎么读？

- **工具无错误**：函数接受了参数并执行，不等于参数符合题意。
- **任务成功**：最终答案和必要工具步骤通过原规则校验。
- 工具调用很多不一定更好；不会做时澄清，比瞎编答案更合理。
- 六个示例不能当作总体成功率；模型尚存在参数填写、跨题型泛化等问题。


In [6]:
print("类别\t调用尝试\t工具无错误\t任务成功\t停止原因")
for trace in traces:
    valid = sum("error" not in c["result"] for c in trace["calls"])
    print(f'{labels[trace["kind"]]}\t{len(trace["calls"])}\t{valid}\t'
          f'{bool(trace["metrics"]["success"])}\t{trace["stop"]}')
assert len(traces) == 6 and all(t["runtime"] == "ocean-openai-agents-v1" for t in traces)
assert all(len(t["calls"]) <= 4 and len(t["rounds"]) <= 6 for t in traces)
print("\n完整输入/输出 tokens 与工具轨迹：", OUTPUT)


类别	调用尝试	工具无错误	任务成功	停止原因
计算	1	1	False	clarify
检索与引用	1	1	True	final
检索后计算	3	3	True	final
错误恢复	1	1	False	clarify
信息不足时澄清	0	0	False	clarify
无需工具直接回答	0	0	True	final

完整输入/输出 tokens 与工具轨迹： /home/anhuang/OceanHeart/artifacts/agent/notebook-sdk-20260927-152635-804324


## 5. 自己出一道题

修改下面的 `QUESTION` 后运行。仍使用验证侧本地资料库，因此不保证能找到任意主题；不联网，也没有人工指定标准答案。每次运行是独立问题，不延续上一题历史。原始输出会保留，不自动打成功标签。


In [7]:
QUESTION = "船速为12 km/h，航行3 h，航程是多少km？请调用工具计算。"
custom_trace = await asyncio.to_thread(run_case, QUESTION)
show_trace(custom_trace)
atomic_json(OUTPUT / ("custom-" + datetime.now().strftime("%H%M%S-%f") + ".json"),
            {"question": QUESTION, "trace": custom_trace})



第 1 轮模型输出：
<plan>按已知数值与单位计算。</plan><tool_call>{"name": "marine_calculate", "arguments": {"operation": "convert", "value": 12.0, "from_unit": "km/h"}}</tool_call>
实际调用： {"name": "marine_calculate", "arguments": {"operation": "convert", "value": 12.0, "from_unit": "km/h"}}
工具结果： {"error": "操作或参数不正确"}

第 2 轮模型输出：
<plan>按已知数值与单位计算。</plan><tool_call>{"name": "marine_calculate", "arguments": {"operation": "distance", "speed": 12.0, "from_unit": "km/h"}}</tool_call>
实际调用： {"name": "marine_calculate", "arguments": {"operation": "distance", "speed": 12.0, "from_unit": "km/h"}}
工具结果： {"error": "操作或参数不正确"}

第 3 轮模型输出：
<final>操作或参数不正确</final>

最终回答： 操作或参数不正确
停止原因： final ；调用尝试： 2


## 结束

观察重点：模型选对工具了吗？参数与题目一致吗？读懂工具结果了吗？失败后是否修正／澄清？

关闭 Notebook 内核可释放显存。此文件不训练模型、不上传 SwanLab；完整模型训练仍走原训练器。

SDK 循环职责参考：[OpenAI 官方运行说明](https://developers.openai.com/api/docs/guides/agents/running-agents)。
